In [ ]:
!pip install rouge-score
!pip install transformers datasets sentencepiece
!pip install -U transformers accelerate
!pip install evaluate

import re
import unicodedata
import pandas as pd
import numpy as np
import argparse
import torch
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from rouge_score import rouge_scorer as rouge_lib
from transformers.optimization import Adafactor

## Preprocessing

train = pd.read_csv('Train.csv')
val   = pd.read_csv('Val.csv')
#test  = pd.read_csv('/content/Test.csv')

ID_COL       = "ID"
QUESTION_COL = "input"
ANSWER_COL   = "output"
SUBSET_COL   = "subset"

SUBSET_TO_LANGUAGE = {
    "Eng_Uga": "english",
    "Eng_Gha": "english",
    "Eng_Eth": "english",
    "Eng_Ken": "english",
    "Aka_Gha": "akan",
    "Lug_Uga": "luganda",
    "Swa_Ken": "swahili",
    #"Amh_Eth": "amharic",
}

SUBSET_TO_COUNTRY = {
    "Eng_Uga": "Uganda",
    "Eng_Gha": "Ghana",
    "Eng_Eth": "Ethiopia",
    "Eng_Ken": "Kenya",
    "Aka_Gha": "Ghana",
    "Lug_Uga": "Uganda",
    "Swa_Ken": "Kenya",
    #"Amh_Eth": "Ethiopia",
}

LOW_RESOURCE_SUBSETS = {#"Amh_Eth",
                        "Swa_Ken",
                        "Eng_Ken"}

LANGUAGE_PREFIXES = {
    "english":  "Answer this health question: ",
    "akan":     "Bua saa safoɔ a ɛfa apɔwmuden ho: ",
    "luganda":  "Ddamu ekibuuzo kino ky'obulamu: ",
    "swahili":  "Jibu swali hili la afya: ",
    #"amharic":  "ይህንን የጤና ጥያቄ ይመልሱ: ",
    "default":  "Answer this health question: ",
}

def normalize_text(text: str) -> str:
    """Unicode NFC normalization + whitespace cleanup."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_language(subset: str) -> str:
    return SUBSET_TO_LANGUAGE.get(str(subset).strip(), "english")

def add_language_prefix(question: str, subset: str) -> str:
    """Prepend a language-specific instruction.
    Helps mT5 condition output language on input."""
    lang   = get_language(subset)
    prefix = LANGUAGE_PREFIXES.get(lang, LANGUAGE_PREFIXES["default"])
    return prefix + normalize_text(question)

def clean_df(df: pd.DataFrame, is_test: bool = False) -> pd.DataFrame:
    df = df.copy()

    # Normalize text
    df[QUESTION_COL] = df[QUESTION_COL].fillna("").apply(normalize_text)
    if not is_test:
        df[ANSWER_COL] = df[ANSWER_COL].fillna("").apply(normalize_text)

    # Normalize subset column
    df[SUBSET_COL] = df[SUBSET_COL].fillna("unknown").str.strip()

    # Drop rows with empty input/output (train/val only)
    if not is_test:
        before = len(df)
        df = df[df[QUESTION_COL].str.len() > 2].copy()
        df = df[df[ANSWER_COL].str.len() > 2].copy()
        after = len(df)
        if before != after:
            print(f"  Dropped {before - after} empty rows")

    # Remove duplicates (276 found in EDA)
    if not is_test:
        before = len(df)
        df = df.drop_duplicates(subset=[QUESTION_COL, ANSWER_COL]).reset_index(drop=True)
        after = len(df)
        if before != after:
            print(f"  Removed {before - after} duplicate Q-A pairs")

    # Build input_text: language prefix + question
    df["input_text"] = df.apply(
        lambda r: add_language_prefix(r[QUESTION_COL], r[SUBSET_COL]), axis=1
    )

    return df

def augment_train(df: pd.DataFrame) -> pd.DataFrame:
    """
    Duplicate low-resource subsets to balance training distribution.
    Low-resource: Amh_Eth (1845), Swa_Ken (2070), Eng_Ken (2080)
    vs high-resource: Eng_Uga (7624).

    Also: English subsets share 494 cross-country questions.
    We keep all of them — the model sees different country contexts
    for the same question, which acts as implicit augmentation.
    """
    aug_parts = [df]

    for subset in LOW_RESOURCE_SUBSETS:
        subset_df = df[df[SUBSET_COL] == subset].copy()
        if len(subset_df) > 0:
            aug_parts.append(subset_df)
            print(f"  Augmented {subset}: +{len(subset_df)} rows (1× duplication)")

    augmented = pd.concat(aug_parts, ignore_index=True)
    augmented = augmented.sample(frac=1, random_state=42).reset_index(drop=True)
    return augmented

train = train[train[SUBSET_COL] != 'Amh_Eth']
train.info()

train = clean_df(train, is_test=False)
val   = clean_df(val,   is_test=False)

train_aug = augment_train(train)
print(f"  Train: {len(train):,} → {len(train_aug):,} after augmentation")
print(f"  Subset counts:\n{train_aug[SUBSET_COL].value_counts().to_string()}")

train_aug.head()

In [ ]:
## Training
MODEL_NAME = "google/mt5-base"

OUTPUT_DIR     = "mt5_health_qa"
TOKENIZER_PATH = MODEL_NAME

# From the data analysis
# Input:  avg 15 words, p95 = ~40 words. This means 128 tokens enough for the inputs
# Output: p99 = 359 words. 512 tokens is enough for all subsets
MAX_INPUT_LENGTH  = 64
MAX_TARGET_LENGTH = 170

BATCH_SIZE        = 32
GRAD_ACCUM_STEPS  = 4
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 5
WARMUP_RATIO      = 0.05
WEIGHT_DECAY      = 0.01
LABEL_SMOOTHING   = 0.1
BF16              = True
SAVE_TOTAL_LIMIT  = 3
EVAL_STEPS        = 1500
LOGGING_STEPS     = 100
SEED              = 42
NUM_BEAMS         = 5
NO_REPEAT_NGRAM   = 3
LENGTH_PENALTY    = 1.2    # slightly favour longer answers (health info is detailed)
EARLY_STOPPING    = True
MAX_NEW_TOKENS    = 170

In [ ]:
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model",       default=MODEL_NAME)
    p.add_argument("--output_dir",  default=OUTPUT_DIR)
    p.add_argument("--epochs",      type=int,   default=NUM_EPOCHS)
    p.add_argument("--batch_size",  type=int,   default=BATCH_SIZE)
    p.add_argument("--lr",          type=float, default=LEARNING_RATE)
    p.add_argument("--max_input",   type=int,   default=MAX_INPUT_LENGTH)
    p.add_argument("--max_target",  type=int,   default=MAX_TARGET_LENGTH)
    p.add_argument("--no_bf16",     action="store_true")
    p.add_argument("--resume",      type=str,   default=None)
    args, unknown = p.parse_known_args()
    return args

In [ ]:
def make_tokenize_fn(tokenizer, max_input, max_target):
    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input_text"],
            max_length=max_input,
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=batch[ANSWER_COL],
            max_length=max_target,
            truncation=True,
            padding=False,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return tokenize

def competition_score(rouge1: float, rougeL: float, llm: float = 0.0) -> float:
    """Competition weighted metric: 0.37*R1 + 0.37*RL + 0.26*LLM"""
    return 0.5 * rouge1 + 0.5 * rougeL

def make_compute_metrics(tokenizer):
    rouge_metric = evaluate.load("rouge")

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred

        # 1. CRITICAL FIX: Replace negative values (like -100) with pad token
        # and cast to standard integers to prevent C++ OverflowErrors
        predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
        predictions = predictions.astype(np.int64) # Force standard integer spacing

        # 2. Cleanly decode predictions
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

        # 3. Cleanly decode labels, ignoring pad/mask tokens
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        labels = labels.astype(np.int64)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # 4. Strip whitespace
        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [ref.strip() for ref in decoded_labels]

        # 5. Compute vectorized metrics across the entire batch instantly
        results = rouge_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=False
        )

        r1 = results["rouge1"]
        rl = results["rougeL"]

        return {
            "rouge1"  : r1,
            "rougeL"  : rl,
            "combined": competition_score(r1, rl),  # 0.5*R1 + 0.5*RL
        }

    return compute_metrics

In [ ]:
args = parse_args()
set_seed(SEED)

train_ds = Dataset.from_pandas(train_aug[["input_text", ANSWER_COL]])
val_ds   = Dataset.from_pandas(val[["input_text", ANSWER_COL]].sample(frac=0.5, random_state=SEED))

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(args.model)
model     = AutoModelForSeq2SeqLM.from_pretrained(args.model) #The model should be loaded with the default fp32 and not changed to fp/bf16
model.config.use_cache = False #setting to True is imcompatible with gradient_checkpointing - error to note

In [ ]:
# Tokenise
tokenize_fn = make_tokenize_fn(tokenizer, args.max_input, args.max_target)
train_tok = train_ds.map(tokenize_fn, batched=True,
                          remove_columns=train_ds.column_names,
                          desc="Tokenising train")
val_tok   = val_ds.map(tokenize_fn,   batched=True,
                        remove_columns=val_ds.column_names,
                        desc="Tokenising val")

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if not args.no_bf16 else None,
)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = args.output_dir,
    num_train_epochs            = args.epochs,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    gradient_checkpointing      = True,
    optim                       = "adafactor",
    learning_rate               = args.lr,
    lr_scheduler_type           = "constant",
    warmup_ratio                = 0,
    weight_decay                = WEIGHT_DECAY,
    #label_smoothing_factor      = LABEL_SMOOTHING,
    bf16                        = BF16 and not args.no_bf16,
    predict_with_generate       = True,
    generation_max_length       = MAX_NEW_TOKENS,
    generation_num_beams        = 1,
    eval_strategy               = "no",
    #eval_steps                  = EVAL_STEPS,
    save_strategy               = "no",
    #save_steps                  = EVAL_STEPS,
    logging_steps               = LOGGING_STEPS,
    load_best_model_at_end      = True,
    metric_for_best_model       = "combined",   # 0.37*R1 + 0.37*RL
    greater_is_better           = True,
    save_total_limit            = SAVE_TOTAL_LIMIT,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    seed                        = SEED,
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = make_compute_metrics(tokenizer),
    #callbacks        = [EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
print("Val dataset size:", len(val_tok))
print("Train dataset size:", len(train_tok))
print("Steps per epoch:", len(train_tok) // (BATCH_SIZE * GRAD_ACCUM_STEPS))
print("Total steps:", (len(train_tok) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS)
print("Eval every:", EVAL_STEPS, "steps")
print("Num evals total:", ((len(train_tok) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS) // EVAL_STEPS)
print("generation_num_beams in training_args:", training_args.generation_num_beams)
print("bf16:", training_args.bf16)
print("gradient_checkpointing:", training_args.gradient_checkpointing)

In [ ]:
trainer.train(resume_from_checkpoint=None)

In [ ]:
# Save model
trainer.save_model(args.output_dir)
#tokenizer.save_pretrained(args.output_dir)
#print(f"\nBest model saved to {args.output_dir}")

In [ ]:
# Final eval
eval_results = trainer.evaluate()
r1 = eval_results.get("eval_rouge1", 0)
rl = eval_results.get("eval_rougeL", 0)
print(f"Final val ROUGE")
print(f"ROUGE-1:  {r1:.4f}")
print(f"ROUGE-L:  {rl:.4f}")
print(f"Combined: {competition_score(r1, rl):.4f}")